In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import to_categorical
import tarfile
import pickle
import numpy as np

2026-11-04 21:44:47.266714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793825087.280492  896789 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793825087.284427  896789 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793825087.296306  896789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793825087.296320  896789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793825087.296322  896789 computation_placer.cc:177] computation placer alr

In [2]:


def load_cifar10(tar_path='cifar-10-batches-py.tar.gz'):
    with tarfile.open(tar_path, 'r:gz') as tar:
        def get_batch(name):
            member = [m for m in tar.getmembers() if m.name.endswith(name)][0]
            return pickle.load(tar.extractfile(member), encoding='bytes')

        # Load 5 training batches
        x_tr, y_tr = [], []
        for i in range(1, 6):
            b = get_batch(f'data_batch_{i}')
            x_tr.append(b[b'data'])
            y_tr.extend(b[b'labels'])

        # Load test batch
        b_test = get_batch('test_batch')
        x_te, y_te = b_test[b'data'], b_test[b'labels']

    # Reshape from flat vectors (3072,) -> (32, 32, 3) image format
    X_train = np.vstack(x_tr).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_train = np.array(y_tr).reshape(-1, 1)

    X_test = x_te.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test = np.array(y_te).reshape(-1, 1)

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = load_cifar10('cifar-10-python.tar.gz')

In [3]:
#step 2 - tf.data pipeline: resize to 224x224 one batch at a time (keeps X_train/X_test as 32x32, saves RAM)
BATCH = 64

def prep(img, label):
    img = tf.image.resize(img, (224, 224))  # stays float, 0-255 (Rescaling layer in the model handles scaling)
    return img, label[0]                    # y is (N,1) -> scalar int label

def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(10000)
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, shuffle=True)
test_ds = make_ds(X_test, y_test)

I0000 00:00:1793825121.315812  896789 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9055 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:27:00.0, compute capability: 7.5


In [4]:
#step 3 - Load MobileNetV2 without top classification head
from tensorflow.keras.applications import MobileNetV2

#load MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet',
                         include_top=False, #without top classification head
                         input_shape=(224, 224, 3))

base_model.trainable = False  #going with freezed pretrained layers (for now)

In [5]:
#step 4 - classification head. Only quantizeml-supported layers: Rescaling, GAP, Dense, Dropout.
#No softmax (not supported) -> the model outputs logits and the loss uses from_logits=True.
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1/127.5, offset=-1, input_shape=(224, 224, 3)),  # raw 0-255 -> [-1, 1] inside the model
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10),
])
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [6]:
#step 5 - train the head (base frozen)
history = model.fit(train_ds, validation_data=test_ds, epochs=10)
print(model.evaluate(test_ds))

Epoch 1/10


I0000 00:00:1793825124.810840  896904 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1793825126.184913  896905 service.cc:152] XLA service 0x7d7c44391860 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793825126.184949  896905 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2026-11-04 21:45:26.191734: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1793825126.280745  896905 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


782/782 [==============================] - 41s 49ms/step - loss: 0.7522 - accuracy: 0.7455 - val_loss: 0.4787 - val_accuracy: 0.8332
Epoch 2/10
782/782 [==============================] - 37s 48ms/step - loss: 0.5715 - accuracy: 0.8056 - val_loss: 0.4543 - val_accuracy: 0.8400
Epoch 3/10
782/782 [==============================] - 37s 48ms/step - loss: 0.5232 - accuracy: 0.8226 - val_loss: 0.4478 - val_accuracy: 0.8437
Epoch 4/10
782/782 [==============================] - 37s 47ms/step - loss: 0.4967 - accuracy: 0.8297 - val_loss: 0.4417 - val_accuracy: 0.8479
Epoch 5/10
782/782 [==============================] - 37s 48ms/step - loss: 0.4745 - accuracy: 0.8372 - val_loss: 0.4451 - val_accuracy: 0.8467
Epoch 6/10
782/782 [==============================] - 37s 48ms/step - loss: 0.4646 - accuracy: 0.8408 - val_loss: 0.4323 - val_accuracy: 0.8513
Epoch 7/10
782/782 [==============================] - 37s 48ms/step - loss: 0.4481 - accuracy: 0.8461 - val_loss: 0.4218 - val_accuracy: 0.8533
Epo

In [7]:
#step 6 - fine-tune (optional): unfreeze the base, recompile with a low learning rate
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history_fine = model.fit(train_ds, validation_data=test_ds, epochs=5)
print(model.evaluate(test_ds))

Epoch 1/5
782/782 [==============================] - 122s 131ms/step - loss: 0.9054 - accuracy: 0.7177 - val_loss: 1.1916 - val_accuracy: 0.6353
Epoch 2/5
782/782 [==============================] - 102s 130ms/step - loss: 0.5288 - accuracy: 0.8270 - val_loss: 1.0796 - val_accuracy: 0.6483
Epoch 3/5
782/782 [==============================] - 102s 131ms/step - loss: 0.4128 - accuracy: 0.8631 - val_loss: 0.6733 - val_accuracy: 0.7840
Epoch 4/5
782/782 [==============================] - 102s 131ms/step - loss: 0.3347 - accuracy: 0.8881 - val_loss: 0.5170 - val_accuracy: 0.8331
Epoch 5/5
157/157 [==============================] - 6s 39ms/step - loss: 0.4260 - accuracy: 0.8605
[0.4259553849697113, 0.8604999780654907]


In [8]:
#calibration samples for quantization: 16 batches of 64 = 1024 raw 0-255 images
samples = np.concatenate([x.numpy() for x, _ in train_ds.take(16)]).astype(np.float32)

2026-11-04 22:00:43.425128: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
import tf_keras as keras
from tf_keras.applications import MobileNetV2

inputs = keras.Input((224, 224, 3))
x = model.layers[0](inputs)                                    # Rescaling
base = MobileNetV2(include_top=False, weights=None, input_tensor=x)   # flat layers
base.set_weights(model.layers[1].get_weights())                # copy trained weights
x = base.output
for layer in model.layers[2:]:                                 # GAP, Dense, Dropout, Dense
    x = layer(x)
flat_model = keras.Model(inputs, x)

print([type(l).__name__ for l in flat_model.layers[:4]])

['InputLayer', 'Rescaling', 'Conv2D', 'BatchNormalization']


In [11]:
xs, ys = [], []
for xb, yb in test_ds.take(32):  # 32 batches x 64 = 2048
    xs.append(xb.numpy()); ys.append(yb.numpy())
x = np.concatenate(xs)[:2000].astype(np.float32)
y = np.concatenate(ys)[:2000]

# 1. Is the flattened model still accurate?
flat_pred = flat_model.predict(x, verbose=0).argmax(axis=1)
print("flat float  :", np.mean(flat_pred == y))

# 2. Full 8-bit quantization (weights and activations)
from quantizeml.models import QuantizationParams, quantize
q8 = quantize(flat_model, qparams=QuantizationParams(), samples=samples)
q8_pred = q8.predict(x, verbose=0).argmax(axis=1)
print("quantized 8-bit:", np.mean(q8_pred == y))

2026-11-04 22:05:49.991144: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


flat float  : 0.8675
1024/1024 [==============================] - 23s 21ms/step
quantized 8-bit: 0.7745


In [14]:
from cnn2snn import convert
akida_model = convert(q8)

In [16]:
import numpy as np
import akida

# Conversion check, then accuracy of float vs quantized vs Akida on 2000 test images.
akida_model.summary()
print("input shape :", akida_model.input_shape)
print("output shape:", akida_model.output_shape)

xs, ys = [], []
for xb, yb in test_ds.take(32):  # 32 batches x 64 = 2048
    xs.append(xb.numpy()); ys.append(yb.numpy())
x = np.concatenate(xs)[:2000].astype(np.float32)
y = np.concatenate(ys)[:2000]

float_pred = model.predict(x, verbose=0).argmax(axis=1)
quant_pred = q8.predict(x, verbose=0).argmax(axis=1)
akida_pred = np.asarray(akida_model.forward(x.astype(np.uint8))).reshape(len(x), -1).argmax(axis=1)
print("float    :", np.mean(float_pred == y))
print("quantized:", np.mean(quant_pred == y))
print("akida    :", np.mean(akida_pred == y))
print("akida vs quantized agreement:", np.mean(akida_pred == quant_pred))

print("devices:", akida.devices())

                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[224, 224, 3]  [1, 1, 10]    1          66    
________________________________________________

______________________________________________________________________________
Layer (type)                               Output shape    Kernel shape     

================== SW/input_quantizer-dequantizer (Software) =================

input_quantizer (Quantizer)                [224, 224, 3]   N/A              
______________________________________________________________________________
Conv1 (InputConv2D)                        [112, 112, 32]  (3, 3, 3, 32)    
______________________________________________________________________________
expanded_conv_depthwise (DepthwiseConv2D)  [112, 112, 32]  (3, 3, 32, 1)    
______________________________________________________________________________
expanded_conv_project (Conv2D)             [112, 112,

2026-11-04 22:13:01.396785: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


float    : 0.8675
quantized: 0.7745
akida    : 0.7705
akida vs quantized agreement: 0.915
devices: []


In [ ]:
model.save("mobilenetv2_cifar10.h5")

In [ ]:
q8.save("mobilenetv2_cifar10q8.h5")

In [17]:
akida_model.save('akida_4bit.fbz')